In [ ]:
import numpy as jnp
import matplotlib.pyplot as plt

## Spectral method 
x_lb = 0 
x_rb = 1
d = 2
n = 4
N_x =2**n
L = x_rb - x_lb
dx = L/N_x
y_lb = x_lb
y_ub = x_rb
N_y = N_x
## 3D Grid
xs, ys, zs = jnp.meshgrid(
    jnp.linspace(x_lb, x_rb, N_x, endpoint=False), 
    jnp.linspace(x_lb, x_rb, N_x, endpoint=False), 
    jnp.linspace(x_lb, x_rb, N_x, endpoint=False),
    indexing='ij'
)

## Diffusion Coefficient Matrix A, 
# div(A grad u) = f
A = jnp.array([
    [3.0, 1.0, 0.5], 
    [1.0, 3.0, 1.0],
    [0.5, 1.0, 3.0]
])



In [ ]:
# True Solution: u(x,y,z) = sin(2pi x) sin(2pi y) sin(2pi z)
def u_true_func(x, y, z):
    return jnp.sin(2*jnp.pi*x) * jnp.sin(2*jnp.pi*y) * jnp.sin(2*jnp.pi*z)

# Corresponding RHS: f = div(A grad u)
def f_rhs_func(x, y, z):
    # Diagonal parts: -4pi^2 (A00 + A11 + A22) * u
    term_diag = -4 * jnp.pi**2 * (A[0,0] + A[1,1] + A[2,2]) * u_true_func(x, y, z)
    
    # Off-diagonal parts
    term_off = 0
    cx, cy, cz = jnp.cos(2*jnp.pi*x), jnp.cos(2*jnp.pi*y), jnp.cos(2*jnp.pi*z)
    sx, sy, sz = jnp.sin(2*jnp.pi*x), jnp.sin(2*jnp.pi*y), jnp.sin(2*jnp.pi*z)
    
    # u_xy, u_xz, u_yz terms
    u_xy = 4 * jnp.pi**2 * cx * cy * sz
    u_xz = 4 * jnp.pi**2 * cx * sy * cz
    u_yz = 4 * jnp.pi**2 * sx * cy * cz
    
    term_off += 2 * A[0,1] * u_xy
    term_off += 2 * A[0,2] * u_xz
    term_off += 2 * A[1,2] * u_yz
    
    return (term_diag + term_off)


In [ ]:
def spectral_eigenvalues(N, L=1.0):
    """Eigenvalues of the 1D derivative operator with periodic condition."""
    k = jnp.fft.fftfreq(N, d=L/N) * 2j * jnp.pi  # frequency vector
    return k  

In [ ]:

"""Constructs the 1D DFT matrix of size N."""
dfmtx = jnp.fft.fft(jnp.eye(N_x))#/jnp.sqrt(N_x)

"""Constructs the 2D DFT matrix of size N x N as a Kronecker product."""
FG = jnp.kron(dfmtx, dfmtx)
GF = jnp.kron(
    (jnp.conj(dfmtx).T)/N_x, 
    (jnp.conj(dfmtx).T)/N_x, 
)

In [ ]:
### 3D FFT Method

def solver_Elliptic_3D_fft(f, A, N, dx): 
    # Evaluate function on grid
    f_values = f(xs, ys, zs)

    # 1. Forward 3D FFT
    f_h = jnp.fft.fftn(f_values)
    
    # 2. Construct Wavenumbers
    # Get 1D wavenumbers
    wave = spectral_eigenvalues(N, dx*N) 
    
    # Create 3D grids of wavenumbers (kx, ky, kz)
    k_x, k_y, k_z = jnp.meshgrid(wave, wave, wave, indexing='ij')
    
    # Stack for vectorized computation: Shape (3, N, N, N)
    K_vec = jnp.array([k_x, k_y, k_z])
    
    # 3. Compute the denominator: sum(A_ij * ki * kj)
    # Using einsum for efficient calculation of A_ij * k_i * k_j
    denom = jnp.einsum('mn, m..., n... -> ...', A, K_vec, K_vec)
    
    # 4. Inversion (Handle Zero Mode)
    # Avoid division by zero at k=(0,0,0)
    denom[0,0,0] = 1.0
    u_h = f_h / denom
    
    # 5. Inverse 3D FFT
    return jnp.real(jnp.fft.ifftn(u_h))


In [ ]:
# Kronecker-based Method

def solver_Elliptic_3D_FG(f, A, N): 
    
    # 1D DFT matrix
    dfmtx = jnp.fft.fft(jnp.eye(N))
    
    # 3D DFT Matrix via Kronecker: F x F x F
    FG = jnp.kron(jnp.kron(dfmtx, dfmtx), dfmtx)
    
    # Inverse 3D DFT Matrix
    dfmtx_inv = jnp.conj(dfmtx).T / N
    GF = jnp.kron(jnp.kron(dfmtx_inv, dfmtx_inv), dfmtx_inv)

    # Diagonal eigenvalue matrix for 1D derivative
    D_diag = spectral_eigenvalues(N)
    D = jnp.diag(D_diag)
    I = jnp.eye(N)
    
    # The spectral operator L is sum of A_ij * (T_i x T_j x T_k)
    # where T is D if the index matches, else I.
    
    # Initialize sparse/dense matrix
    Elliptic_spec = jnp.zeros((N**3, N**3), dtype=complex)
    
    # Helper for 3-way kron
    def kron3(a, b, c):
        return jnp.kron(jnp.kron(a, b), c)

    for i in range(3):
        for j in range(3):
            val = A[i, j]
            # Identify components [x_op, y_op, z_op]
            # Default to Identity
            mats = [I, I, I] 
            
            # Update for derivative i
            if i == 0: mats[0] = D
            elif i == 1: mats[1] = D
            elif i == 2: mats[2] = D
            
            # Multiply for derivative j (order doesn't matter for diagonal D matrices)
            if j == 0: mats[0] = mats[0] @ D
            elif j == 1: mats[1] = mats[1] @ D
            elif j == 2: mats[2] = mats[2] @ D
            
            # Add to total operator
            Elliptic_spec += val * kron3(mats[0], mats[1], mats[2])

    f_flatten = f(xs, ys, zs).flatten()
    f_h = FG @ f_flatten

    # 2. Inversion (Solve linear system)
    Elliptic_spec[0, 0] = 1.0
    inverse_Elliptic = jnp.linalg.inv(Elliptic_spec)
  

    # 3. Inverse Transform
    u_flatten = GF @ (inverse_Elliptic @ f_h)

    return u_flatten.reshape(N, N, N).real



In [ ]:
u_spec = solver_Elliptic_3D_fft(f_rhs_func, A=A, N=N_x, dx=dx)
u_theo = solver_Elliptic_3D_FG(f_rhs_func, A, N_x)
#plt.figure(figsize=(10, 5))
#plt.subplot(1, 3, 1)
#plt.imshow(u_spec.real, extent=[x_lb, x_rb, y_lb, y_ub], origin='lower')
#plt.title("u_spectral")
#plt.subplot(1, 3, 2)
#plt.imshow(u_theo.real, extent=[x_lb, x_rb, y_lb, y_ub], origin='lower')
#plt.title("u_theoretical")
#plt.subplot(1, 3, 3)
#plt.imshow(abs(u_spec-u_theo), extent=[x_lb, x_rb, y_lb, y_ub], origin='lower')
#plt.title("abs_error")
#plt.colorbar()

# Plotting
z_idx = N_x // 4
z_val = zs[0,0,z_idx]

plt.figure(figsize=(15, 5))



plt.subplot(1, 3, 1)
plt.imshow(u_theo[:, :, z_idx], origin='lower', extent=[x_lb, x_rb, x_lb, x_rb], vmin=-1, vmax =1)
plt.title("Reference Solver")
plt.colorbar()

plt.subplot(1, 3, 2)
plt.imshow(u_spec[:, :, z_idx], origin='lower', extent=[x_lb, x_rb, x_lb, x_rb], vmin=-1, vmax=1)
plt.title("Classical Solver")
plt.colorbar()

plt.subplot(1, 3, 3)
plt.imshow(jnp.abs(u_spec[:, :, z_idx] - u_theo[:, :, z_idx]), origin='lower', extent=[x_lb, x_rb, x_lb, x_rb])
plt.title("Error |Classical - Reference|")
plt.colorbar()

plt.tight_layout()
plt.show()

In [ ]:
normalized_u_spec = u_spec / jnp.linalg.norm(u_spec)
normalized_u_theo = u_theo/ jnp.linalg.norm(u_theo)
error = jnp.linalg.norm(u_spec - u_theo)/jnp.linalg.norm(u_theo)
error

In [ ]:
import numpy as np
from scipy.linalg import sqrtm, dft
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFT
from qiskit_aer import AerSimulator
def make_unitary(A):
    N = A.shape[0]
    sqrt_term = sqrtm(np.eye(N) - A.conj().T @ A)
    return np.block([
        [A,         sqrt_term],
        [sqrt_term, -A       ]
    ])


# Helper for 3-way kron
def kron3(a, b, c):
    return jnp.kron(jnp.kron(a, b), c)

def build_elliptic_inverse(N, A):

    D_diag = spectral_eigenvalues(N)
    D = jnp.diag(D_diag)
    I = jnp.eye(N)
 # Initialize sparse/dense matrix
    Elliptic_spec = jnp.zeros((N**3, N**3), dtype=complex)
    
    for i in range(3):
        for j in range(3):
            val = A[i, j]
            # Identify components [x_op, y_op, z_op]
            # Default to Identity
            mats = [I, I, I] 
            
            # Update for derivative i
            if i == 0: mats[0] = D
            elif i == 1: mats[1] = D
            elif i == 2: mats[2] = D
            
            # Multiply for derivative j (order doesn't matter for diagonal D matrices)
            if j == 0: mats[0] = mats[0] @ D
            elif j == 1: mats[1] = mats[1] @ D
            elif j == 2: mats[2] = mats[2] @ D
            
            # Add to total operator
            Elliptic_spec += val * kron3(mats[0], mats[1], mats[2])
    Elliptic_spec[0, 0] = 1.0                     # avoid zero inversion
    return np.linalg.inv(Elliptic_spec)


# ── Quantum circuit ────────────────────────────────────────────────────────────

def build_elliptic_circuit(inverse_Elliptic, n):

    total_qubits = 3* n + 1
    # Normalise so that A / alpha has spectral norm ≤ 1
    alpha    = np.linalg.norm(inverse_Elliptic)
    A_normed = inverse_Elliptic / alpha

    U = make_unitary(A_normed)
    
    qft  = QFT(n)
    qft1 = QFT(n)
    qft2 = QFT(n)

    qc = QuantumCircuit(total_qubits)
    qc.append(qft,          list(range(n)))
    qc.append(qft1,         list(range(n, 2 * n)))
    qc.append(qft2,         list(range(2 * n, 3 * n)))
    qc.unitary(U,           list(range(total_qubits)))
    qc.append(qft.inverse(),  list(range(n)))
    qc.append(qft1.inverse(), list(range(n, 2 * n)))
    qc.append(qft2.inverse(), list(range(2 * n, 3 * n)))

    return qc, alpha


def extract_unitary(qc):
    """Simulates the circuit and returns its full unitary matrix."""
    print("  Simulating unitary...")
    qc_sim = qc.copy()
    qc_sim.save_unitary()
    sim    = AerSimulator(method="unitary")
    qc_sim = transpile(qc_sim, sim)
    result = sim.run(qc_sim).result()
    return np.array(result.get_unitary(qc_sim))



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors

def plot_solutions_and_errors(xs, ys, zs, u_theo, u_spec, u_quantum, label='', save_path=None):
    """
    3D Voxel visualization for Elliptic solutions.
    Unified 'turbo' colormap, high-vibrancy alpha, and fixed isometric rotation.
    """
    # 1. Prepare data
    u_theo_r    = np.array(u_theo).real
    u_spec_r    = np.array(u_spec).real
    u_quantum_r = np.array(u_quantum).real

    # Side length 8 for 512 total points
    N_side = int(np.round(u_theo_r.size**(1/3)))
    
    # 2. Reshape to 3D cube (8, 8, 8)
    # Using 'ij' meshgrid logic, the reshape preserves the spatial orientation
    u_theo_3d    = u_theo_r.reshape(N_side, N_side, N_side)
    u_spec_3d    = u_spec_r.reshape(N_side, N_side, N_side)
    u_quantum_3d = u_quantum_r.reshape(N_side, N_side, N_side)

    err_FG = np.abs(u_spec_3d - u_theo_3d)
    err_QC = np.abs(u_quantum_3d - u_theo_3d)

    fig = plt.figure(figsize=(20, 10))
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.1, wspace=0.2)
    
    # Scaling limits
    vmin_u, vmax_u = u_theo_3d.min(), u_theo_3d.max()
    vmax_e = max(err_FG.max(), err_QC.max()) or 1e-10

    def render_voxels(ax, data_3d, vmin, vmax, title):
        # Full 8x8x8 dense grid
        filled = np.ones(data_3d.shape, dtype=bool)
        
        # Color mapping: Turbo for high brightness and contrast
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        cmap = plt.get_cmap('turbo') 
        
        facecolors = cmap(norm(data_3d))
        facecolors[..., 3] = 0.65  # Vivid alpha for transparency
        
        # Plotting the voxels with thin white borders for "glow"
        ax.voxels(filled, facecolors=facecolors, edgecolors='white', 
                  linewidth=0.08, shade=True)
        
        ax.set_title(title, fontsize=15, pad=15, fontweight='bold')
        
        # --- ROTATION SETTINGS ---
        # elev: vertical angle, azim: horizontal rotation
        # -45 azim puts the (0,0) origin at the bottom-front
        ax.view_init(elev=25, azim=-45)
        
        ax.set_axis_off() 
        ax.set_box_aspect([1, 1, 1]) 
        
        return plt.cm.ScalarMappable(norm=norm, cmap=cmap)

    # --- Row 1: Solutions ---
    ax1 = fig.add_subplot(gs[0, 0], projection='3d')
    render_voxels(ax1, u_theo_3d, vmin_u, vmax_u, 'Reference')

    ax2 = fig.add_subplot(gs[0, 1], projection='3d')
    render_voxels(ax2, u_spec_3d, vmin_u, vmax_u, 'Classical (FG)')

    ax3 = fig.add_subplot(gs[0, 2], projection='3d')
    sm_u = render_voxels(ax3, u_quantum_3d, vmin_u, vmax_u, 'Quantum')
    
    # Solution Colorbar
    cax_u = fig.add_axes([0.92, 0.55, 0.015, 0.35])
    cb_u = fig.colorbar(sm_u, cax=cax_u)
    cb_u.set_label('Amplitude ($u$)', fontsize=12, fontweight='bold')

    # --- Row 2: Errors ---
    ax5 = fig.add_subplot(gs[1, 1], projection='3d')
    render_voxels(ax5, err_FG, 0, vmax_e, 'Classical Error')

    ax6 = fig.add_subplot(gs[1, 2], projection='3d')
    sm_e = render_voxels(ax6, err_QC, 0, vmax_e, 'Quantum Error')

    # Error Colorbar
    cax_e = fig.add_axes([0.92, 0.1, 0.015, 0.35])
    cb_e = fig.colorbar(sm_e, cax=cax_e)
    cb_e.set_label('Absolute Error', fontsize=12, fontweight='bold')

    if label:
        fig.suptitle(f"Elliptic Solver 3D Voxel Analysis: {label}", fontsize=22, y=0.98)

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=200)
    plt.show()

In [ ]:
import os
import h5py
import numpy as np

# ── Output directory ───────────────────────────────────────────────────────────
out_dir = "results_elliptic_A_configs3D"
os.makedirs(out_dir, exist_ok=True)

A_configs = [
        {
        'label': 'Identity_3x3', 
        'matrix': jnp.eye(3)
    },
    {
        'label': 'Custom_3_1_05_3x3', 
        'matrix': np.array([
            [3.0, 1.0, 0.5],
            [1.0, 3.0, 1.0],
            [0.5, 1.0, 3.0]
        ])
    },
    {
        'label': 'Custom_10_1_1_3x3', 
        'matrix': np.array([
            [10.0, 0.0, 0.0],
            [0.0,  1.0, 0.0],
            [0.0,  0.0, 1.0]
        ])
    },
    {
        'label': 'Custom_1_100_1_3x3', 
        'matrix': np.array([
            [1.0, 0.0,   0.0],
            [0.0, 100.0, 0.0],
            [0.0, 0.0,   1.0]
        ])
    },
    {
        'label': 'Custom_1_100_01_3x3', 
        'matrix': np.array([
            [1.0, 0.0,   0.0],
            [0.0, 100.0, 0.0],
            [0.0, 0.0,   0.1]
        ])
    },
    {
        'label': 'Custom_1_1_100000_3x3', 
        'matrix': np.array([
            [1.0, 0.0, 0.0],
            [0.0, 1.0, 0.0],
            [0.0, 0.0, 100000.0]
        ])
    },
]
plt.rcParams.update({'font.size': 22})

for cfg in A_configs:
    label = cfg['label']
    print(f"\n{'='*55}")
    print(f"  A configuration : {label}")
    print(f"  Matrix:\n{cfg['matrix']}")
    print(f"{'='*55}")

    # ── Update global A ──
    A = cfg['matrix']

    # ── Build inverse elliptic operator ──────────────────────────────
    print("\n  Building inverse elliptic operator...")
    inverse_Elliptic = build_elliptic_inverse(N_x, A)

    # ── Classical solution ────────────────────────────────────────────
    print("\n  Running classical solver...")
    u_spec = solver_Elliptic_3D_fft(f_rhs_func, A, N_x, dx)
    u_theo = solver_Elliptic_3D_FG(f_rhs_func, A, N_x)
    
    # Ensure they are 3D for error calculation and plotting
    u_spec = u_spec.reshape(N_x, N_x, N_x)
    u_theo = u_theo.reshape(N_x, N_x, N_x)
    
    error_classical = float(np.linalg.norm(u_spec - u_theo) / np.linalg.norm(u_theo))

    # ── Quantum solution ──────────────────────────────────────────────
    print("\n  Computing quantum solution...")
    qc, alpha = build_elliptic_circuit(inverse_Elliptic, n)
    mat = extract_unitary(qc)

    block_size = 2 ** (3 * n)
    leading_block = mat[:block_size, :block_size]
    
    f_vec = f_rhs_func(xs, ys, zs).flatten()
    f_norm = np.linalg.norm(f_vec)
    f_normalized = f_vec / f_norm
    
    u_quantum_vec = leading_block @ (alpha * f_normalized) * f_norm
    u_quantum = u_quantum_vec.reshape(N_x, N_x, N_x)
    
    error_quantum = float(np.linalg.norm(u_quantum - u_theo) / np.linalg.norm(u_theo))

    print(f"\n  {'─'*43}")
    print(f"  Classical error : {error_classical:.6e}")
    print(f"  Quantum error   : {error_quantum:.6e}")

    # ── Save results to HDF5 file ─────────────────────────────────────
    h5_path = os.path.join(out_dir, f"{label}_results.h5")
    with h5py.File(h5_path, 'w') as hf:
        # Saving 3D datasets
        hf.create_dataset('u_theoretical', data=u_theo.real)
        hf.create_dataset('u_classical',   data=u_spec.real)
        hf.create_dataset('u_quantum',     data=u_quantum.real)
        hf.create_dataset('A_matrix',      data=cfg['matrix'])
        
        # Metadata attributes
        hf.attrs['error_classical'] = error_classical
        hf.attrs['error_quantum']   = error_quantum
        hf.attrs['label']           = label
    
    print(f"  H5 Data saved:  {h5_path}")

    # ── Save solutions / errors figure ────────────────────────────────
    path_fig = os.path.join(out_dir, f"{label}_solutions_errors.png")
    plot_solutions_and_errors(xs, ys, zs, u_theo, u_spec, u_quantum,
                              label=label, save_path=path_fig)
    print(f"  Figure saved : {path_fig}")

print(f"\n{'='*55}")
print(f"  All done! Results saved in: '{out_dir}/'")
print(f"{'='*55}")